In [4]:
import os
import pickle
import numpy as np
import pandas as pd
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression



In [5]:
# ---------------------------------------------------------------------
# 1. Cargar los datos
# ---------------------------------------------------------------------
data_path = "../data/processed/tienda_tecnologica_limpio.csv"
df = pd.read_csv(data_path)
print(f"Filas: {df.shape[0]} | Columnas: {df.shape[1]}")
print("Columnas:", df.columns.tolist())
 


Filas: 1019 | Columnas: 14
Columnas: ['id_venta', 'fecha', 'producto', 'marca', 'categoria', 'precio', 'cantidad', 'garantia_meses', 'stock_dias', 'calificacion', 'canal', 'devolucion', 'total', 'opinion_usuario']


In [6]:
# 2. Generar la columna 'sentimiento' a partir de 'calificacion'
#
#    'calificacion' llega sucia: valores como '1,00' (coma decimal),
#    negativos (-1 a -5, tratados como error de captura) y 'Desconocido'
#    (calificación faltante, se descarta esa fila).
#
#    Regla: calificacion >= 3  -> POSITIVO (1)
#           calificacion <  3  -> NEGATIVO (0)
# ---------------------------------------------------------------------
def limpiar_calificacion(valor):
    if valor == "Desconocido":
        return np.nan
    valor_str = str(valor).replace(",", ".")
    return abs(float(valor_str))
 
df["calificacion_limpia"] = df["calificacion"].apply(limpiar_calificacion)
df = df.dropna(subset=["calificacion_limpia"])
 
df["sentimiento"] = (df["calificacion_limpia"] >= 3).astype(int)
 
print("\nDistribución de sentimiento (1=POSITIVO, 0=NEGATIVO):")
print(df["sentimiento"].value_counts())



Distribución de sentimiento (1=POSITIVO, 0=NEGATIVO):
sentimiento
1    548
0    411
Name: count, dtype: int64


In [7]:

# ---------------------------------------------------------------------
# 3. Vectorizar el texto de las opiniones (TF-IDF)
# ---------------------------------------------------------------------
X_raw = df["opinion_usuario"].fillna("").astype(str)
y = df["sentimiento"].values
 
vectorizer = TfidfVectorizer(max_features=5000)
X_vectorized = vectorizer.fit_transform(X_raw).toarray()
 
X_train, X_test, y_train, y_test = train_test_split(
    X_vectorized, y, test_size=0.2, random_state=42, stratify=y
)
 


In [8]:
# ---------------------------------------------------------------------
# 4. Definir y entrenar la red neuronal
# ---------------------------------------------------------------------
model = keras.Sequential([
    layers.Input(shape=(X_train.shape[1],)),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(1, activation="sigmoid")
])
 
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)
 
history = model.fit(
    X_train,
    y_train,
    epochs=10,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)


E0000 00:00:1788815652.225787   19659 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Epoch 1/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.5612 - loss: 0.6900 - val_accuracy: 0.5649 - val_loss: 0.6766
Epoch 2/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5710 - loss: 0.6808 - val_accuracy: 0.5974 - val_loss: 0.6742
Epoch 3/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5726 - loss: 0.6782 - val_accuracy: 0.5649 - val_loss: 0.6710
Epoch 4/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5742 - loss: 0.6800 - val_accuracy: 0.5974 - val_loss: 0.6701
Epoch 5/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.5742 - loss: 0.6767 - val_accuracy: 0.5909 - val_loss: 0.6709
Epoch 6/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5759 - loss: 0.6764 - val_accuracy: 0.6039 - val_loss: 0.6722
Epoch 7/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5726 - loss: 0.6818 - val_accuracy: 0.6039 - val_loss: 0.6712
Epoch 8/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5726 - loss: 0.6761 - val_accuracy: 0.6039 - val_loss

In [9]:
# ---------------------------------------------------------------------
# 5. Evaluar en datos de prueba
# ---------------------------------------------------------------------
loss, accuracy = model.evaluate(X_test, y_test)
print(f"\nPérdida (Loss): {loss:.4f}")
print(f"Precisión (Accuracy): {accuracy:.4f}")
 


6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.5573 - loss: 0.6918 

Pérdida (Loss): 0.6918
Precisión (Accuracy): 0.5573


In [10]:

modelo_tradicional = LogisticRegression(max_iter=1000)
modelo_tradicional.fit(X_train, y_train)
accuracy_tradicional = modelo_tradicional.score(X_test, y_test)

print(f"\nComparación de modelos:")
print(f"Regresión Logística (tradicional): {accuracy_tradicional:.4f}")
print(f"Red Neuronal (Deep Learning):      {accuracy:.4f}")


Comparación de modelos:
Regresión Logística (tradicional): 0.5521
Red Neuronal (Deep Learning):      0.5573


In [11]:
# ---------------------------------------------------------------------
# 6. Guardar el modelo y el vectorizador
# ---------------------------------------------------------------------
os.makedirs("../models", exist_ok=True)
model.save("../models/neural_network.h5")
with open("../models/tfidf_vectorizer_dl.pkl", "wb") as f:
    pickle.dump(vectorizer, f)
 
print("\n¡Archivos guardados en 'models/neural_network.h5' y 'models/tfidf_vectorizer_dl.pkl'!")
 



¡Archivos guardados en 'models/neural_network.h5' y 'models/tfidf_vectorizer_dl.pkl'!
